In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print("TensorFlow version:", tf.__version__)
print("Google Colab setup complete!")


In [ ]:

# Download the Jena Climate dataset
!wget -q https://s3.amazonaws.com/keras-datasets/jena_climate_2009_2016.csv.zip

# Extract the dataset
!unzip -o -q jena_climate_2009_2016.csv.zip

# Load the dataset
df = pd.read_csv("jena_climate_2009_2016.csv")

# Display the first five rows
df.head()


In [ ]:

# Extract temperature as our prediction target
temperature = df["T (degC)"].to_numpy()

# Remove the date column and retain weather measurements
raw_data = df.drop(columns=["Date Time"]).to_numpy()

# Define chronological training and validation boundaries
num_train_samples = int(0.5 * len(raw_data))
num_val_samples = int(0.25 * len(raw_data))
num_test_samples = len(raw_data) - num_train_samples - num_val_samples

print("Training samples:", num_train_samples)
print("Validation samples:", num_val_samples)
print("Testing samples:", num_test_samples)


In [ ]:

# Calculate mean and standard deviation using training data only
mean = raw_data[:num_train_samples].mean(axis=0)
std = raw_data[:num_train_samples].std(axis=0)

# Normalize all weather measurements
raw_data = (raw_data - mean) / std

print("Normalization complete!")
print("Number of weather features:", raw_data.shape[1])


In [ ]:

# Define time-series parameters
sampling_rate = 6
sequence_length = 120
delay = sampling_rate * (sequence_length + 24 - 1)
batch_size = 256

# Create training dataset
train_dataset = keras.utils.timeseries_dataset_from_array(
    raw_data[:-delay],
    targets=temperature[delay:],
    sampling_rate=sampling_rate,
    sequence_length=sequence_length,
    shuffle=True,
    batch_size=batch_size,
    start_index=0,
    end_index=num_train_samples
)

# Create validation dataset
val_dataset = keras.utils.timeseries_dataset_from_array(
    raw_data[:-delay],
    targets=temperature[delay:],
    sampling_rate=sampling_rate,
    sequence_length=sequence_length,
    shuffle=True,
    batch_size=batch_size,
    start_index=num_train_samples,
    end_index=num_train_samples + num_val_samples
)

# Create testing dataset
test_dataset = keras.utils.timeseries_dataset_from_array(
    raw_data[:-delay],
    targets=temperature[delay:],
    sampling_rate=sampling_rate,
    sequence_length=sequence_length,
    shuffle=True,
    batch_size=batch_size,
    start_index=num_train_samples + num_val_samples
)

print("Time-series datasets created successfully!")


In [ ]:

# Calculate common-sense baseline MAE
def evaluate_naive_method(dataset):
    total_absolute_error = 0
    total_samples = 0

    for samples, targets in dataset:
        # Most recent temperature, converted back to Celsius
        predictions = samples[:, -1, 1] * std[1] + mean[1]

        total_absolute_error += np.sum(
            np.abs(predictions.numpy() - targets.numpy())
        )
        total_samples += targets.shape[0]

    return total_absolute_error / total_samples

# Evaluate baseline
val_mae = evaluate_naive_method(val_dataset)
test_mae = evaluate_naive_method(test_dataset)

print(f"Validation MAE: {val_mae:.2f} °C")
print(f"Test MAE: {test_mae:.2f} °C")


In [ ]:

# Build the first GRU model
inputs = keras.Input(shape=(sequence_length, raw_data.shape[-1]))

x = layers.GRU(32, recurrent_dropout=0.25)(inputs)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(1)(x)

gru_model = keras.Model(inputs, outputs)

gru_model.compile(
    optimizer="rmsprop",
    loss="mse",
    metrics=["mae"]
)

# Save the best model based on validation MAE
callbacks = [
    keras.callbacks.ModelCheckpoint(
        "best_gru_model.keras",
        save_best_only=True,
        monitor="val_mae",
        mode="min"
    ),
    keras.callbacks.EarlyStopping(
        monitor="val_mae",
        patience=5,
        restore_best_weights=True
    )
]

print("GRU model created successfully!")


In [ ]:

# Train the GRU model
history_gru = gru_model.fit(
    train_dataset,
    epochs=20,
    validation_data=val_dataset,
    callbacks=callbacks
)

# Identify the best epoch and validation MAE
best_epoch = np.argmin(history_gru.history["val_mae"]) + 1
best_val_mae = min(history_gru.history["val_mae"])

print(f"Best Epoch: {best_epoch}")
print(f"Best Validation MAE: {best_val_mae:.4f} °C")


In [ ]:

# Experiment 2: GRU with 64 units
inputs = keras.Input(shape=(sequence_length, raw_data.shape[-1]))

x = layers.GRU(64, recurrent_dropout=0.25)(inputs)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(1)(x)

gru64_model = keras.Model(inputs, outputs)

gru64_model.compile(
    optimizer="rmsprop",
    loss="mse",
    metrics=["mae"]
)

callbacks_gru64 = [
    keras.callbacks.ModelCheckpoint(
        "best_gru64_model.keras",
        save_best_only=True,
        monitor="val_mae",
        mode="min"
    ),
    keras.callbacks.EarlyStopping(
        monitor="val_mae",
        patience=5,
        restore_best_weights=True
    )
]

history_gru64 = gru64_model.fit(
    train_dataset,
    epochs=20,
    validation_data=val_dataset,
    callbacks=callbacks_gru64
)

best_epoch_gru64 = np.argmin(history_gru64.history["val_mae"]) + 1
best_val_mae_gru64 = min(history_gru64.history["val_mae"])

print(f"Best Epoch: {best_epoch_gru64}")
print(f"Best Validation MAE: {best_val_mae_gru64:.4f} °C")


In [ ]:

# Experiment 3: LSTM with 32 units
inputs = keras.Input(shape=(sequence_length, raw_data.shape[-1]))

x = layers.LSTM(32, recurrent_dropout=0.25)(inputs)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(1)(x)

lstm_model = keras.Model(inputs, outputs)

lstm_model.compile(
    optimizer="rmsprop",
    loss="mse",
    metrics=["mae"]
)

callbacks_lstm = [
    keras.callbacks.ModelCheckpoint(
        "best_lstm_model.keras",
        save_best_only=True,
        monitor="val_mae",
        mode="min"
    ),
    keras.callbacks.EarlyStopping(
        monitor="val_mae",
        patience=5,
        restore_best_weights=True
    )
]

history_lstm = lstm_model.fit(
    train_dataset,
    epochs=20,
    validation_data=val_dataset,
    callbacks=callbacks_lstm
)

best_epoch_lstm = np.argmin(history_lstm.history["val_mae"]) + 1
best_val_mae_lstm = min(history_lstm.history["val_mae"])

print(f"Best Epoch: {best_epoch_lstm}")
print(f"Best Validation MAE: {best_val_mae_lstm:.4f} °C")


In [ ]:

# Experiment 4: Stacked GRU model
inputs = keras.Input(shape=(sequence_length, raw_data.shape[-1]))

x = layers.GRU(
    32,
    recurrent_dropout=0.25,
    return_sequences=True
)(inputs)

x = layers.GRU(
    32,
    recurrent_dropout=0.25
)(x)

x = layers.Dropout(0.5)(x)
outputs = layers.Dense(1)(x)

stacked_gru_model = keras.Model(inputs, outputs)

stacked_gru_model.compile(
    optimizer="rmsprop",
    loss="mse",
    metrics=["mae"]
)

callbacks_stacked = [
    keras.callbacks.ModelCheckpoint(
        "best_stacked_gru_model.keras",
        save_best_only=True,
        monitor="val_mae",
        mode="min"
    ),
    keras.callbacks.EarlyStopping(
        monitor="val_mae",
        patience=5,
        restore_best_weights=True
    )
]

history_stacked = stacked_gru_model.fit(
    train_dataset,
    epochs=20,
    validation_data=val_dataset,
    callbacks=callbacks_stacked
)

best_epoch_stacked = np.argmin(
    history_stacked.history["val_mae"]
) + 1

best_val_mae_stacked = min(
    history_stacked.history["val_mae"]
)

print(f"Best Epoch: {best_epoch_stacked}")
print(f"Best Validation MAE: {best_val_mae_stacked:.4f} °C")


In [ ]:

# Experiment 5: Conv1D + GRU
inputs = keras.Input(shape=(sequence_length, raw_data.shape[-1]))

x = layers.Conv1D(
    filters=32,
    kernel_size=5,
    activation="relu"
)(inputs)

x = layers.MaxPooling1D(pool_size=2)(x)

x = layers.GRU(
    32,
    recurrent_dropout=0.25
)(x)

x = layers.Dropout(0.5)(x)
outputs = layers.Dense(1)(x)

conv_gru_model = keras.Model(inputs, outputs)

conv_gru_model.compile(
    optimizer="rmsprop",
    loss="mse",
    metrics=["mae"]
)

callbacks_conv_gru = [
    keras.callbacks.ModelCheckpoint(
        "best_conv_gru_model.keras",
        save_best_only=True,
        monitor="val_mae",
        mode="min"
    ),
    keras.callbacks.EarlyStopping(
        monitor="val_mae",
        patience=5,
        restore_best_weights=True
    )
]

history_conv_gru = conv_gru_model.fit(
    train_dataset,
    epochs=20,
    validation_data=val_dataset,
    callbacks=callbacks_conv_gru
)

best_epoch_conv_gru = np.argmin(
    history_conv_gru.history["val_mae"]
) + 1

best_val_mae_conv_gru = min(
    history_conv_gru.history["val_mae"]
)

print(f"Best Epoch: {best_epoch_conv_gru}")
print(f"Best Validation MAE: {best_val_mae_conv_gru:.4f} °C")


In [ ]:

# Load the best saved models
best_gru32 = keras.models.load_model("best_gru_model.keras")
best_gru64 = keras.models.load_model("best_gru64_model.keras")
best_conv_gru = keras.models.load_model("best_conv_gru_model.keras")

# Evaluate on the test dataset
test_gru32 = best_gru32.evaluate(test_dataset, verbose=0)[1]
test_gru64 = best_gru64.evaluate(test_dataset, verbose=0)[1]
test_conv_gru = best_conv_gru.evaluate(test_dataset, verbose=0)[1]

print(f"GRU 32 Test MAE: {test_gru32:.4f} °C")
print(f"GRU 64 Test MAE: {test_gru64:.4f} °C")
print(f"Conv1D + GRU Test MAE: {test_conv_gru:.4f} °C")


In [ ]:

# Final validation MAE comparison chart
models = [
    "Baseline", "GRU 32", "GRU 64",
    "LSTM 32", "Stacked GRU", "Conv1D + GRU"
]

validation_mae = [
    2.4400, 2.2673, 2.2667,
    2.3024, 2.3083, 2.2687
]

plt.figure(figsize=(10, 5))
bars = plt.bar(models, validation_mae)

plt.title("Validation MAE by Model")
plt.ylabel("Mean Absolute Error (°C)")
plt.xlabel("Model Architecture")
plt.ylim(0, 2.7)
plt.xticks(rotation=20)

for bar, value in zip(bars, validation_mae):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        value + 0.03,
        f"{value:.4f}",
        ha="center"
    )

plt.tight_layout()
plt.savefig("assignment3_validation_comparison.png", dpi=300)
plt.show()


# Assignment 3: Time-Series Weather Forecasting Using Deep Learning

## Objective

The objective of this assignment was to evaluate different deep learning architectures for time-series weather forecasting and determine which approach produced the most accurate temperature predictions. Five neural network configurations were compared against a common-sense baseline using mean absolute error (MAE).

## Methodology

The Jena Climate dataset was used to predict temperature 24 hours into the future based on five days of historical weather measurements. The dataset was divided chronologically into 50% training, 25% validation, and 25% testing data. Weather measurements were normalized using statistics calculated from the training data.

Five models were evaluated: a 32-unit GRU, a 64-unit GRU, a 32-unit LSTM, a stacked GRU, and a combined Conv1D and GRU model. Dropout and early stopping were used to reduce overfitting. Models were compared using their lowest validation MAE, and the three strongest configurations were evaluated on the test dataset.

## Results

| Model | Best Epoch | Validation MAE (°C) | Test MAE (°C) |
|---|---:|---:|---:|
| Common-Sense Baseline | N/A | 2.4400 | 2.6200 |
| GRU (32 Units) | 9 | 2.2673 | 2.4362 |
| GRU (64 Units) | 5 | 2.2667 | 2.4246 |
| LSTM (32 Units) | 10 | 2.3024 | Not Evaluated |
| Stacked GRU (32 + 32) | 7 | 2.3083 | Not Evaluated |
| Conv1D + GRU | 5 | 2.2687 | 2.4633 |

The 64-unit GRU achieved the lowest validation MAE of 2.2667°C and the lowest test MAE of 2.4246°C. This represented approximately a 7.46% reduction in test error compared with the baseline.

## Discussion

All five neural network configurations outperformed the common-sense baseline on validation data, demonstrating that deep learning can identify useful patterns in historical weather measurements.

Increasing the GRU from 32 to 64 units produced only a marginal improvement in validation accuracy. Similarly, adding a second GRU layer did not improve performance, suggesting that additional model complexity was not beneficial in these experiments.

The LSTM model performed slightly worse than both single-layer GRU configurations. The combined Conv1D and GRU architecture produced competitive validation results but had a higher test error than either single-layer GRU model.

Although the 64-unit GRU performed best, the differences among the strongest models were small. Additional training runs would help determine whether these differences are consistent.

## Conclusion

The results demonstrate that recurrent neural networks can improve temperature forecasting compared with a simple baseline approach. The 64-unit GRU achieved the best overall performance, with a test MAE of 2.4246°C.

However, increasing architectural complexity did not consistently improve predictive accuracy. These findings suggest that selecting an appropriate model architecture and controlling overfitting are more important than simply increasing the number of layers or units.